# 00_part2b - Download ALL buckets' images in one pass

Images are **shared across all 6 buckets** (raw product photos, ASIN-unique),
so there is no need to run `00_part2` six times. This notebook unions every
bucket's clean ASIN list, downloads only the images **not already on disk**,
and prints a per-bucket coverage table at the end.

Does NOT modify `00_part2`. Re-running is safe and incremental (skips images
already present in `data/images/`).

HOW TO RUN: Restart Kernel, then Run All (top to bottom). Cells share state
and MUST run in order.


## (1) Config

In [1]:
import os
from pathlib import Path


def _find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data_preparation').exists() and (p / 'code').exists():
            return p
    raise RuntimeError(
        'Cannot find project root (need both data_preparation/ and code/).'
    )


PROJECT_ROOT = _find_project_root()

# All 6 buckets. Images are shared, so we sweep them all in one run.
BUCKETS = [
    'women_top1_dream_pairs', 'women_top2_idifu', 'women_other',
    'men_top1_bruno_marc',    'men_top2_nike',    'men_other',
]

IMG_DIR = PROJECT_ROOT / 'data' / 'images'      # shared (ASIN-unique)
IMG_DIR.mkdir(parents=True, exist_ok=True)

MAX_RETRIES = 3
TIMEOUT_SEC = 10
SLEEP_SEC   = 0.2


def clean_file(bucket: str) -> Path:
    return (PROJECT_ROOT / 'data' / bucket /
            'amzn_shoes_monthly_diffs_ffill_fixed_splits' /
            'all_products_clean.parquet')


print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'IMG_DIR      : {IMG_DIR}')
print(f'Buckets      : {len(BUCKETS)}')


PROJECT_ROOT : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate
IMG_DIR      : /Users/yvony/PycharmProjects/Amazon_demand_analysis_seperate/data/images
Buckets      : 6


## (2) Union ASIN + image URL across all buckets

In [2]:
import pandas as pd

bucket_asins = {}          # bucket -> set of ASINs that have an image URL
frames = []

print('Loading clean parquets per bucket:')
for b in BUCKETS:
    cf = clean_file(b)
    if not cf.exists():
        print(f'  [skip] {b}: missing all_products_clean.parquet '
              f'(run 00_part1 for this bucket first)')
        bucket_asins[b] = set()
        continue
    df = pd.read_parquet(cf, columns=['ASIN', 'image'])
    df = df[['ASIN', 'image']].drop_duplicates('ASIN').dropna(subset=['image'])
    bucket_asins[b] = set(df['ASIN'])
    frames.append(df)
    print(f'  [ok]   {b}: {len(df):,} ASINs with image URL')

# One row per ASIN (image URL is static per product)
asin_urls = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates('ASIN')
    .reset_index(drop=True)
)
print(f'\nUnique ASINs across all buckets: {len(asin_urls):,}')


Loading clean parquets per bucket:
  [ok]   women_top1_dream_pairs: 442 ASINs with image URL
  [ok]   women_top2_idifu: 178 ASINs with image URL
  [ok]   women_other: 4,216 ASINs with image URL
  [ok]   men_top1_bruno_marc: 165 ASINs with image URL
  [ok]   men_top2_nike: 103 ASINs with image URL
  [ok]   men_other: 1,466 ASINs with image URL

Unique ASINs across all buckets: 6,570


## (3) Skip images already on disk

In [3]:
already_done = {f[:-4] for f in os.listdir(IMG_DIR) if f.endswith('.jpg')}
to_download = asin_urls[~asin_urls['ASIN'].isin(already_done)].reset_index(drop=True)

print(f'Already on disk : {len(already_done):,} images')
print(f'To download     : {len(to_download):,} images')
print(f'Target total    : {len(asin_urls):,} images')


Already on disk : 6,128 images
To download     : 442 images
Target total    : 6,570 images


## (4) Download only the missing images

In [4]:
import time
import requests
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

success, failed = [], []

for _, row in tqdm(to_download.iterrows(), total=len(to_download),
                   desc='Downloading'):
    asin = row['ASIN']
    url = row['image']
    save_path = IMG_DIR / f'{asin}.jpg'

    downloaded = False
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT_SEC)
            if resp.status_code == 200:
                img = Image.open(BytesIO(resp.content)).convert('RGB')
                img.save(save_path, 'JPEG')
                success.append(asin)
                downloaded = True
                break
        except Exception:
            pass
        time.sleep(SLEEP_SEC)
    if not downloaded:
        failed.append(asin)

print(f'\nDownloaded : {len(success):,}')
print(f'Failed     : {len(failed):,}')
if failed:
    print('First few failed ASINs:', failed[:10])


Downloading:   0%|          | 0/442 [00:00<?, ?it/s]


Downloaded : 442
Failed     : 0


## (5) Per-bucket coverage

In [5]:
on_disk = {f[:-4] for f in os.listdir(IMG_DIR) if f.endswith('.jpg')}

print('=' * 60)
print('PER-BUCKET IMAGE COVERAGE')
print('=' * 60)
print(f'{"bucket":<26}{"with_url":>10}{"on_disk":>10}{"cover":>8}')
for b in BUCKETS:
    want = bucket_asins.get(b, set())
    have = len(want & on_disk)
    pct = (have / len(want) * 100) if want else 0.
    print(f'{b:<26}{len(want):>10,}{have:>10,}{pct:>7.1f}%')

total_size = sum(
    os.path.getsize(IMG_DIR / f)
    for f in os.listdir(IMG_DIR) if f.endswith('.jpg')
) / 1e6
print(f'\nTotal images on disk: {len(on_disk):,}  ({total_size:.1f} MB)')


PER-BUCKET IMAGE COVERAGE
bucket                      with_url   on_disk   cover
women_top1_dream_pairs           442       442  100.0%
women_top2_idifu                 178       178  100.0%
women_other                    4,216     4,216  100.0%
men_top1_bruno_marc              165       165  100.0%
men_top2_nike                    103       103  100.0%
men_other                      1,466     1,466  100.0%

Total images on disk: 6,570  (1413.1 MB)
